# AIC System — Notebook 05: Run Queries & Export Submission

**Purpose:** Load the pre-built FAISS index and run KIS / Q&A queries,
then export BTC-standard submission CSV files.

**Prerequisites:**
- Notebook `kaggle_01_build_index.ipynb` has been run → `aic-hcmc-indexes` dataset available
- Query JSON file uploaded to Kaggle or defined inline below

**Outputs** (in `/kaggle/working/submission/`):
- `submission_kis.csv` — Dạng 1 results
- `submission_qa.csv` — Dạng 2 results (answer column filled in Sprint 4)
- `results.json` — Full results for inspection

In [ ]:
# ============================================================
# CELL 1: Setup — clone GitHub repo
# ============================================================
import subprocess, sys, os

GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"  # ← change
REPO_DIR = "/kaggle/working/AIC_System"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", f"{REPO_DIR}/requirements.txt"], check=True)
print("Setup complete.")

In [ ]:
# ============================================================
# CELL 2: Configure Paths
# ============================================================
from pathlib import Path

# Dataset slugs (last part of Kaggle URL)
INDEX_DATASET = "aic-hcmc-indexes"    # ← slug of your indexes dataset

INDEX_DIR    = Path(f"/kaggle/input/{INDEX_DATASET}")
OUTPUT_DIR   = Path("/kaggle/working/submission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify index files
for fname in ["faiss_visual.index", "keyframe_master.parquet"]:
    fpath = INDEX_DIR / fname
    if fpath.exists():
        print(f"  OK: {fname} ({fpath.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  MISSING: {fname}")

print(f"Output: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# CELL 3: Define Queries
# ============================================================
# Option A: Load from Kaggle input dataset
# QUERIES_PATH = "/kaggle/input/aic-queries/queries.json"

# Option B: Define inline for quick testing
import json

QUERIES = [
    {
        "query_id": "q001",
        "type": "textual_kis",
        "text": "Tìm video về một diễn giả mặc áo đỏ phát biểu tại cuộc họp báo ngoài trời, phía sau có nhiều cây xanh"
    },
    {
        "query_id": "q002",
        "type": "textual_kis",
        "text": "Video về cầu thủ bóng đá ghi bàn và ăn mừng trên sân vận động"
    },
    {
        "query_id": "q003",
        "type": "qa",
        "description": "Lễ trao giải âm nhạc với nhiều nghệ sĩ đứng trên sân khấu",
        "question": "Có bao nhiêu người lên sân khấu nhận giải thưởng lớn nhất?"
    },
]

# Save to file for the CLI runner
QUERIES_PATH = str(OUTPUT_DIR / "queries.json")
with open(QUERIES_PATH, "w", encoding="utf-8") as f:
    json.dump(QUERIES, f, ensure_ascii=False, indent=2)

print(f"Saved {len(QUERIES)} queries → {QUERIES_PATH}")

In [ ]:
# ============================================================
# CELL 4: Load Pipeline
# ============================================================
import time
from src.pipeline.retrieval_pipeline import RetrievalPipeline

t0 = time.time()
pipeline = RetrievalPipeline.from_index_dir(
    index_dir=str(INDEX_DIR),
    clip_model="ViT-B-32",
    clip_pretrained="openai",
)
print(f"Pipeline ready in {time.time() - t0:.1f}s")

In [ ]:
# ============================================================
# CELL 5: Run Queries & Collect Results
# ============================================================
from src.evaluation.submission_formatter import SubmissionFormatter
from src.reasoning.query_classifier import QueryClassifier
from src.common.enums import QueryType

formatter  = SubmissionFormatter(output_dir=str(OUTPUT_DIR))
classifier = QueryClassifier()

all_results = []

for query_dict in QUERIES:
    qid   = str(query_dict.get("query_id", "?"))
    qtype = classifier.classify(query_dict)
    t_q   = time.time()

    evidence = pipeline.run(query_dict, query_id=qid)

    if evidence:
        if qtype == QueryType.TEXTUAL_KIS:
            formatter.add_kis(qid, evidence)
        elif qtype == QueryType.QA:
            formatter.add_qa(qid, evidence, answer="")

        all_results.append({
            "query_id": qid, "type": qtype.value,
            "video_id": evidence.video_id, "frame_idx": evidence.frame_idx,
            "pts_time": f"{evidence.pts_time:.2f}s", "score": f"{evidence.confidence:.4f}",
            "latency": f"{time.time()-t_q:.2f}s"
        })
        print(f"  [{qid}] {evidence.video_id} frame_idx={evidence.frame_idx} "
              f"pts={evidence.pts_time:.2f}s score={evidence.confidence:.4f} "
              f"({time.time()-t_q:.2f}s)")
    else:
        print(f"  [{qid}] No result found")

import pandas as pd
pd.DataFrame(all_results)

In [ ]:
# ============================================================
# CELL 6: Save Submission Files
# ============================================================
kis_path  = formatter.save_kis()
qa_path   = formatter.save_qa()
json_path = formatter.save_json()

print("\nSubmission files:")
for p in [kis_path, qa_path, json_path]:
    if p.exists():
        print(f"  {p.name}: {p.stat().st_size / 1024:.1f} KB")

print("\nKIS submission preview:")
import pandas as pd
pd.read_csv(kis_path)